# Test Trained Llama Function Calling Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch
import json
import os

In [ ]:
BASE_MODEL = "meta-llama/Llama-3.2-1B-Instruct"
ADAPTER = "./2_1B_Instruct_xLAM/checkpoint-20"

# Find checkpoint
if os.path.exists(ADAPTER):
    ckpts = sorted([d for d in os.listdir(ADAPTER) if d.startswith('checkpoint')])
    if ckpts:
        ADAPTER = os.path.join(ADAPTER, ckpts[-1])
        print(f"Using: {ADAPTER}")
else:
    print("Available:", [d for d in os.listdir('.') if 'xLAM' in d])

In [ ]:
print("Loading...")
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32, device_map="cpu")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = PeftModel.from_pretrained(base, ADAPTER)
model = model.merge_and_unload()
model.eval()
print("✅ Ready")

Loading...


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

✅ Ready


In [ ]:
def generate(query, tools=None):
    """Generate function call - matches training format exactly."""
    if tools is None:
        tools = [{"name": "get_weather", "parameters": {"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]}}]
    
    tools_json = json.dumps(tools, separators=(',', ':'))
    
    # EXACT training format
    prompt = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{query}\n\nTools: {tools_json}\n\nCalling: "
    
    inputs = tokenizer(prompt, return_tensors="pt")
    prompt_len = len(inputs['input_ids'][0])
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=50,  # Small limit
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    
    # Get only generated text (after prompt)
    result = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    return result

In [ ]:
# Quick test
q = "Weather in NYC?"
print(f"Q: {q}")
print(f"A: {generate(q)}")

Q: Weather in NYC?
A: [{"name":"get_weather","arguments":{"location":"New York City (US)"}}]


In [ ]:
q_1 = "Weather in Boston City?"
print(f"Q: {q_1}")
print(f"A: {generate(q_1)}")

Q: Weather in Boston City?
A: [{"name":"get_weather","arguments":{"location":"Boston City"}},{"description":"Get the current weather for a given location.","returns":{"description":"The current weather information.","type":"array":[{"description":"An array containing an object with 'temperature' and


In [ ]:
q_2 = "Where can I find live giveaways for beta access and games?"
print(f"Q: {q_2}")
print(f"A: {generate(q_2)}")

Q: Where can I find live giveaways for beta access and games?
A: {"description":"Get the current weather in a given location.","parameters":{"location":{"description":"The city or zip code of where to get the weather information.","type":"str"}}}
